In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL = "facebook/nllb-200-distilled-600M"
SAVE_DIR = "/Users/pranav/Music/nllb_offline"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)

tokenizer.save_pretrained(SAVE_DIR)
model.save_pretrained(SAVE_DIR)

print("Offline model saved to:", SAVE_DIR)

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Offline model saved to: /Users/pranav/Music/nllb_offline


In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_DIR = "/Users/pranav/Music/nllb_offline"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    local_files_only=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_DIR,
    local_files_only=True
)

device = "mps" if torch.backends.mps.is_available() else "cpu"
model = model.to(device)

# --------------------------------------------------
# Language map
# --------------------------------------------------
LANG = {
    "si": "sin_Sinhala",   # Sinhala
    "ne": "npi_Deva",      # Nepali
    "te": "tel_Telu",      # Telugu
    "kn": "kan_Knda",      # Kannada
    "ta": "tam_Taml",      # Tamil
    "ml": "mal_Mlym",      # Malayalam
    "hi": "hin_Deva",      # Hindi
    "en": "eng_Latn"       # English
}

# --------------------------------------------------
# Offline translate function
# --------------------------------------------------
def translate(text, src, tgt="en"):
    tokenizer.src_lang = LANG[src]
    inputs = tokenizer(text, return_tensors="pt").to(device)

    forced_bos_token_id = tokenizer.convert_tokens_to_ids(
        f"<{LANG[tgt]}>"
    )

    out = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_length=256
    )

    return tokenizer.batch_decode(out, skip_special_tokens=True)[0]

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

In [3]:
print(translate("ඔබට කොහොමද?", "si"))
print(translate("तपाईंलाई कस्तो छ?", "ne"))
print(translate("మీరు ఎలా ఉన్నారు?", "te"))
print(translate("ನೀವು ಹೇಗಿದ್ದೀರಾ?", "kn"))
print(translate("நீங்கள் எப்படி இருக்கிறீர்கள்?", "ta"))
print(translate("നിങ്ങൾക്ക് സുഖമാണോ?", "ml"))
print(translate("आप कैसे हैं?", "hi"))

How are you?
How are you?
How are you?
How are you?
How are you?
Are you okay?
How are you?
